In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix

# 1-0. 데이터 로드
data_path = '../data/prep/total_data.csv'
data = pd.read_csv(data_path)

# 1-1. 학습 데이터 추출
train_df = data[data['eval_set'] == 'train']

# 학습에 사용하지 않을 식별자 및 중복 컬럼 제외
unused_cols = ['user_id', 'order_id', 'eval_set', 'product_id', 'reordered', 'order_number']
X = train_df.drop(columns=unused_cols)
y = train_df['reordered']

# 1-2. 학습용/검증용 데이터 분리 (8:2)
train_feat, val_feat, train_labels, val_labels = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"학습 피처 형태: {train_feat.shape}, 학습 정답 형태: {train_labels.shape}")
print(f"검증 피처 형태: {val_feat.shape}, 검증 정답 형태: {val_labels.shape}")

In [ ]:
# LightGBM 전용 데이터셋으로 변환
dtrain = lgb.Dataset(train_feat, label=train_labels)
dval = lgb.Dataset(val_feat, label=val_labels, reference=dtrain)

In [ ]:
# 3-1. 파라미터 수정
params = {
    'objective': 'binary',
    'metric': 'auc',             # logloss 대신 AUC를 평가지표로 사용 (불균형에 더 강함)
    'boosting_type': 'gbdt',
    'scale_pos_weight': 9.2,      # 9.2배 가중치 (165,765 / 1,529,168 비율 반영)
    'learning_rate': 0.01,        # 아주 조금씩 신중하게 학습하도록 낮춤
    'num_leaves': 63,             # 모델이 더 복잡한 패턴을 볼 수 있게 키움
    'feature_fraction': 0.7,      # 변수를 조금씩만 써서 과적합 방지
    'bagging_fraction': 0.7,
    'bagging_freq': 5,
    'seed': 42,
    'verbose': -1
}

# 3-2. 모델 학습 (동일)
model_lgb = lgb.train(
    params,
    dtrain,
    num_boost_round=1000,         # 목표는 최소 100~300회 이상 학습하는 것
    valid_sets=[dtrain, dval],
    valid_names=['train', 'valid'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=10)   # 흐름을 더 자주 확인할 수 있게 10으로 변경
    ]
)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

# 1. 검증 데이터(val_feat)로 확률 예측
# LightGBM의 predict는 기본적으로 클래스 1(재구매)에 대한 확률값을 반환
val_probs = model_lgb.predict(val_feat)

# 2. 임계값(Threshold) 설정 
# 로지스틱 회귀와 비교하기 위해 우선 0.2를 적용
custom_threshold = 0.2
val_preds_custom = (val_probs >= custom_threshold).astype(int)

# 3. 혼동 행렬을 이용한 상세 지표 계산
tn, fp, fn, tp = confusion_matrix(val_labels, val_preds_custom).ravel()

sensitivity = tp / (tp + fn)       # Recall
specificity = tn / (tn + fp)
pos_pred_value = tp / (tp + fp)    # Precision
neg_pred_value = tn / (tn + fn)
prevalence = (tp + fn) / (tp + tn + fp + fn)
detection_rate = tp / (tp + tn + fp + fn)
detection_prevalence = (tp + fp) / (tp + tn + fp + fn)
balanced_accuracy = (sensitivity + specificity) / 2

# 4. 결과 출력
print(f"[LightGBM 임계값 {custom_threshold} 적용 결과]")
print(f"Accuracy: {accuracy_score(val_labels, val_preds_custom):.4f}")
print(f"F1-Score: {f1_score(val_labels, val_preds_custom):.4f}")

print("\n[상세 분류 리포트]")
print(f"            Sensitivity : {sensitivity:.5f}")
print(f"            Specificity : {specificity:.5f}")
print(f"         Pos Pred Value : {pos_pred_value:.5f}")
print(f"         Neg Pred Value : {neg_pred_value:.5f}")
print(f"             Prevalence : {prevalence:.5f}")
print(f"         Detection Rate : {detection_rate:.5f}")
print(f"   Detection Prevalence : {detection_prevalence:.5f}")
print(f"      Balanced Accuracy : {balanced_accuracy:.5f}")

print("\n[기본 분류 리포트]")
print(classification_report(val_labels, val_preds_custom, target_names=['미구매(0)', '재구매(1)']))

In [ ]:
# 임계값을 0.4부터 0.8까지 넓게 확인해봅니다.
print("[임계값 재튜닝 결과]")
for t in [0.4, 0.5, 0.6, 0.7, 0.8]:
    preds = (val_probs >= t).astype(int)
    f1 = f1_score(val_labels, preds)
    print(f"Threshold {t:.1f} -> F1-Score: {f1:.4f}")

In [ ]:
import numpy as np
import lightgbm as lgb
from sklearn.metrics import f1_score, classification_report

# 1. 파라미터 고도화 (성능 극대화 설정)
params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'is_unbalance': True,         # 불균형 대응
    'learning_rate': 0.005,       # 학습률을 낮춰서 아주 세밀하게 학습 (기존 0.01)
    'num_leaves': 127,            # 트리를 더 복잡하게 구성 (기존 63)
    'feature_fraction': 0.7,      # 과적합 방지하며 변수 탐색
    'bagging_fraction': 0.7,
    'bagging_freq': 5,
    'min_data_in_leaf': 100,      # 너무 자잘한 패턴에 휘둘리지 않게 설정
    'seed': 42,
    'verbose': -1
}

# 2. 모델 재학습 (반복 횟수 증가)
# 학습 곡선이 계속 우상향했으므로 3000회까지 열어둡니다.
model_lgb_v2 = lgb.train(
    params,
    dtrain,
    num_boost_round=3000,         # 더 긴 호흡으로 학습
    valid_sets=[dtrain, dval],
    valid_names=['train', 'valid'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=100), # 100번 참아줌
        lgb.log_evaluation(period=100)
    ]
)

# 3. 최적의 임계값(Threshold) 자동 탐색 (0.01 단위)
val_probs = model_lgb_v2.predict(val_feat)
thresholds = np.arange(0.5, 0.86, 0.01) # 현재 0.7 근처가 핫하므로 범위를 좁혀 정밀 탐색
best_f1 = 0
best_threshold = 0

print("\n[최적 임계값 탐색 시작]")
for t in thresholds:
    preds = (val_probs >= t).astype(int)
    current_f1 = f1_score(val_labels, preds)
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_threshold = t

print(f"최적 임계값: {best_threshold:.2f}")
print(f"최고 F1-Score: {best_f1:.4f}")

# 4. 최종 결과 출력
final_preds = (val_probs >= best_threshold).astype(int)
print("\n[최종 모델 상세 분류 리포트]")
print(classification_report(val_labels, final_preds, target_names=['미구매(0)', '재구매(1)']))

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, classification_report, confusion_matrix, accuracy_score

# 1. 최종 모델로 확률 예측
val_probs = model_lgb_v2.predict(val_feat)

# 2. 0.01 단위 정밀 스캐닝 (0.60 ~ 0.80)
best_f1 = 0
best_threshold = 0

print(f"{'Threshold':<12} | {'F1-Score':<10}")
print("-" * 25)

for t in np.arange(0.60, 0.81, 0.01):
    preds = (val_probs >= t).astype(int)
    current_f1 = f1_score(val_labels, preds)
    
    # 0.01 단위별 점수 출력
    print(f"{t:<12.2f} | {current_f1:<10.4f}")
    
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_threshold = t

print("-" * 25)
print(f"발견된 최적 임계값: {best_threshold:.2f}")
print(f"최종 최고 F1-Score: {best_f1:.4f}")

# 3. 최적 임계값을 적용한 최종 상세 리포트
final_preds = (val_probs >= best_threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(val_labels, final_preds).ravel()

print("\n" + "="*50)
print(f" [ 최종 모델 성능 리포트 - 임계값 {best_threshold:.2f} ]")
print("="*50)
print(f" Accuracy           : {accuracy_score(val_labels, final_preds):.4f}")
print(f" F1-Score           : {best_f1:.4f}")
print(f" Sensitivity(Recall): {tp / (tp + fn):.5f}  <-- 재구매자 중 맞춘 비율")
print(f" Precision          : {tp / (tp + fp):.5f}  <-- 모델이 '산다'고 한 것 중 맞춘 비율")
print(f" Specificity        : {tn / (tn + fp):.5f}")
print(f" Balanced Accuracy  : {((tp/(tp+fn)) + (tn/(tn+fp)))/2:.5f}")
print("-" * 50)
print("\n[기본 분류 리포트]")
print(classification_report(val_labels, final_preds, target_names=['미구매(0)', '재구매(1)']))

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# 1. 변수 중요도 데이터 추출 (Gain 기준: 모델의 예측력에 기여한 정도)
ftr_importances_values = model_lgb_v2.feature_importance(importance_type='gain')
ftr_importances = pd.Series(ftr_importances_values, index=train_feat.columns)

# 2. 중요도 순으로 정렬하여 상위 20개 추출
ftr_top20 = ftr_importances.sort_values(ascending=False)[:20]

# 3. 시각화 (그래프 그리기)
plt.figure(figsize=(10, 8))
plt.title('LightGBM Feature Importance (Top 20 - Gain)')
sns.barplot(x=ftr_top20, y=ftr_top20.index, palette='viridis')

# 수치 표시
for i, v in enumerate(ftr_top20):
    plt.text(v, i, f' {v:,.0f}', va='center')

plt.xlabel('Importance (Total Gain)')
plt.ylabel('Features')
plt.tight_layout()
plt.show()

# (참고) 전체 순위를 데이터프레임으로 확인하고 싶을 때
importance_df = pd.DataFrame({
    'Feature': train_feat.columns,
    'Importance': ftr_importances_values
}).sort_values(by='Importance', ascending=False)

print(importance_df.head(20))